# Data Structures in Practice: 2D Animation Character Design Lab -- SOLUTIONS

**Series:** Machine Learning from Scratch | Part 1 of 3 -- Lists, Tuples, and Dictionaries

**Document Purpose.** This is the complete solution notebook (Solucionario Oficial) for Part 1. It contains full reference implementations for all code cells, verified execution outputs, and detailed answers with rationales for all Logic Questions (Preguntas Lógicas). It serves as the master key for instructors and for generating automated Moodle evaluation quizzes.

**Context.** A 2D animation studio has registered production data for twenty characters from the animated series *Neon Skies*. The raw dataset contains intentional data quality errors (missing values, inconsistent types, incomplete color palettes). This notebook demonstrates how to inspect, clean, and enrich the dataset using core Python data structures without external libraries, preparing it for Pandas analysis in Part 2 and Linear Regression in Part 3.

**Summary of Intentional Errors and Corrections Applied:**
- Missing `role` values: `Crux` (assigned `"support"`) and `Rex` (assigned `"antagonist"`).
- String `fps` values: `Kael` (`"24"` -> `24`) and `Thorn` (`"12"` -> `12`).
- Missing `has_special_fx` values: `Zorn` and `Rex` (`None` -> `0`).
- Incomplete palettes: `Mira` (+1 color), `Drak` (+2 colors), `Olmo` (+2 colors).
- Derived fields added: `num_colors`, `avg_brightness`, `color_contrast`, `num_states`, `num_frames_total`.

---
## Initial Dataset -- Reference Data

The cell below defines `characters_raw`, the unmodified raw list of twenty character dictionaries containing the intentional errors.

In [1]:
# Dataset: 2D animated series "Neon Skies"
# Raw initial dataset containing intentional data quality errors

characters_raw = [
    {"name": "Lyra",   "role": "protagonist", "palette": [(240,180,90),(80,40,20),(255,255,255)],             "states": ["idle","walk","run","attack","fall"],                      "fps": 24,    "has_special_fx": 1,    "style": "cel_shaded", "screen_time_ratio": 0.38, "production_hours": 142.0},
    {"name": "Kael",   "role": "antagonist",  "palette": [(30,15,60),(180,0,255),(10,10,10)],                 "states": ["idle","walk","attack","taunt","defeat"],                  "fps": "24",  "has_special_fx": 1,    "style": "painted",    "screen_time_ratio": 0.25, "production_hours": 138.5},
    {"name": "Mira",   "role": "support",     "palette": [(200,230,255),(120,180,240)],                       "states": ["idle","walk"],                                             "fps": 12,    "has_special_fx": 0,    "style": "flat",       "screen_time_ratio": 0.12, "production_hours": 48.0},
    {"name": "Zorn",   "role": "antagonist",  "palette": [(50,50,50),(200,10,10),(100,0,0),(255,50,50)],      "states": ["idle","walk","run","attack"],                              "fps": 24,    "has_special_fx": None, "style": "cel_shaded", "screen_time_ratio": 0.18, "production_hours": 115.0},
    {"name": "Fenn",   "role": "support",     "palette": [(180,220,160),(90,140,80),(255,240,200)],           "states": ["idle","walk","emote"],                                    "fps": 12,    "has_special_fx": 0,    "style": "flat",       "screen_time_ratio": 0.08, "production_hours": 52.0},
    {"name": "Seraph", "role": "protagonist", "palette": [(255,220,150),(200,160,80),(255,255,220),(180,120,60)],"states": ["idle","walk","run","attack","special","fall"],          "fps": 24,    "has_special_fx": 1,    "style": "painted",    "screen_time_ratio": 0.30, "production_hours": 168.0},
    {"name": "Crux",   "role": None,          "palette": [(40,40,80),(120,120,200),(200,200,240)],             "states": ["idle","walk","attack"],                                   "fps": 12,    "has_special_fx": 0,    "style": "flat",       "screen_time_ratio": 0.05, "production_hours": 44.0},
    {"name": "Vela",   "role": "support",     "palette": [(255,180,200),(200,100,130),(240,240,255)],         "states": ["idle","walk","emote","special"],                          "fps": 12,    "has_special_fx": 1,    "style": "cel_shaded", "screen_time_ratio": 0.10, "production_hours": 78.0},
    {"name": "Drak",   "role": "antagonist",  "palette": [(20,20,20),(150,0,0)],                              "states": ["idle","walk","run","attack","rage"],                       "fps": 24,    "has_special_fx": 1,    "style": "painted",    "screen_time_ratio": 0.20, "production_hours": 130.0},
    {"name": "Pip",    "role": "support",     "palette": [(255,200,100),(200,150,50),(255,255,180)],          "states": ["idle","walk"],                                             "fps": 8,     "has_special_fx": 0,    "style": "flat",       "screen_time_ratio": 0.04, "production_hours": 30.0},
    {"name": "Asha",   "role": "protagonist", "palette": [(100,200,180),(50,150,130),(200,240,235),(30,100,90)],"states": ["idle","walk","run","attack","special"],                  "fps": 24,    "has_special_fx": 1,    "style": "painted",    "screen_time_ratio": 0.28, "production_hours": 155.0},
    {"name": "Thorn",  "role": "antagonist",  "palette": [(60,30,10),(180,90,20),(100,50,10),(220,120,40)],   "states": ["idle","walk","attack","taunt"],                           "fps": "12",  "has_special_fx": 0,    "style": "cel_shaded", "screen_time_ratio": 0.14, "production_hours": 88.0},
    {"name": "Lune",   "role": "support",     "palette": [(220,220,255),(180,180,240),(255,255,255)],         "states": ["idle","walk","emote"],                                    "fps": 12,    "has_special_fx": 0,    "style": "flat",       "screen_time_ratio": 0.07, "production_hours": 46.0},
    {"name": "Rex",    "role": None,          "palette": [(200,50,50),(240,100,80),(80,20,20)],               "states": ["idle","walk","run","attack"],                              "fps": 24,    "has_special_fx": None, "style": "cel_shaded", "screen_time_ratio": 0.16, "production_hours": 102.0},
    {"name": "Cleo",   "role": "support",     "palette": [(240,210,160),(180,140,90),(255,230,200)],          "states": ["idle","walk","emote","special"],                          "fps": 12,    "has_special_fx": 1,    "style": "cel_shaded", "screen_time_ratio": 0.09, "production_hours": 72.0},
    {"name": "Blaze",  "role": "protagonist", "palette": [(255,100,30),(200,60,10),(255,200,50),(180,40,0)], "states": ["idle","walk","run","attack","special","fall","victory"],   "fps": 24,    "has_special_fx": 1,    "style": "painted",    "screen_time_ratio": 0.35, "production_hours": 185.0},
    {"name": "Nyx",    "role": "antagonist",  "palette": [(10,0,30),(80,0,160),(150,50,255)],                 "states": ["idle","walk","attack","special"],                         "fps": 24,    "has_special_fx": 1,    "style": "painted",    "screen_time_ratio": 0.22, "production_hours": 128.0},
    {"name": "Olmo",   "role": "support",     "palette": [(160,200,120)],                                     "states": ["idle"],                                                   "fps": 8,     "has_special_fx": 0,    "style": "flat",       "screen_time_ratio": 0.03, "production_hours": 18.0},
    {"name": "Sela",   "role": "support",     "palette": [(200,180,220),(160,140,180),(240,220,255)],         "states": ["idle","walk","emote"],                                    "fps": 12,    "has_special_fx": 0,    "style": "flat",       "screen_time_ratio": 0.06, "production_hours": 42.0},
    {"name": "Gorr",   "role": "antagonist",  "palette": [(80,60,40),(160,120,80),(40,30,20),(200,150,100)],  "states": ["idle","walk","run","attack","rage","defeat"],             "fps": 24,    "has_special_fx": 1,    "style": "painted",    "screen_time_ratio": 0.24, "production_hours": 145.0},
]

print(f"Dataset loaded: {len(characters_raw)} characters registered.")
print("Fields in each record:", list(characters_raw[0].keys()))

Dataset loaded: 20 characters registered.
Fields in each record: ['name', 'role', 'palette', 'states', 'fps', 'has_special_fx', 'style', 'screen_time_ratio', 'production_hours']


---
## Part 1: Lists

### Exercise 1.1 -- Accessing the Dataset by Index

**Reference.** A list stores elements in a fixed order. Elements are accessed by their zero-based index: `list[0]` retrieves the first element, `list[-1]` the last. The built-in function `len(list)` returns the total number of elements.

**Example.** The following cell illustrates how to access records in a list of dictionaries by index.

In [2]:
# Example -- do not modify
game_records = [
    {"title": "Celeste",  "genre": "platformer"},
    {"title": "Hades",    "genre": "action"},
    {"title": "Cuphead",  "genre": "run-and-gun"},
]
print("First title:",   game_records[0]["title"])
print("Last title:",    game_records[-1]["title"])
print("Total records:", len(game_records))

First title: Celeste
Last title: Cuphead
Total records: 3


**Task.** Complete the student cell below using `characters_raw`:
- Print the name of the first character in the dataset.
- Print the name of the last character in the dataset.
- Print the name of the character at index 9.
- Print the total number of characters registered.

In [3]:
# Solution 1.1
first_name = characters_raw[0]["name"]
print("First character:", first_name)

last_name = characters_raw[-1]["name"]
print("Last character:", last_name)

tenth_name = characters_raw[9]["name"]
print("Character at index 9:", tenth_name)

total = len(characters_raw)
print("Total characters:", total)

First character: Lyra
Last character: Gorr
Character at index 9: Pip
Total characters: 20


**Logic Question 1.** Based exclusively on the output produced by running the cell above:

a) What is the name of the character at index 9?
b) What is the name of the last character in the dataset?
c) How many characters are registered in total?

**Solution / Respuestas:**
- **a)** `Pip` (index 9 corresponds to the 10th element in zero-based indexing).
- **b)** `Gorr` (retrieved via negative index `characters_raw[-1]["name"]` or `characters_raw[19]["name"]`).
- **c)** `20` characters registered in total (`len(characters_raw)`).

### Exercise 1.2 -- Extracting Fields with List Comprehension

**Reference.** A list comprehension builds a new list by evaluating an expression for each element: `[expression for item in list]`. To include only elements that meet a condition, a filter is appended: `[expression for item in list if condition]`.

**Example.** The following cell extracts all titles and then only the titles belonging to the platformer genre.

In [4]:
# Example -- do not modify
game_records = [
    {"title": "Celeste",       "genre": "platformer"},
    {"title": "Hades",         "genre": "action"},
    {"title": "Cuphead",       "genre": "run-and-gun"},
    {"title": "Shovel Knight", "genre": "platformer"},
]
all_titles  = [r["title"] for r in game_records]
print("All titles:", all_titles)

platformers = [r["title"] for r in game_records if r["genre"] == "platformer"]
print("Platformers only:", platformers)

All titles: ['Celeste', 'Hades', 'Cuphead', 'Shovel Knight']
Platformers only: ['Celeste', 'Shovel Knight']


**Task.** Complete the student cell below:
- Build a list `names` containing the name of every character.
- Build a list `protagonists` containing the names of characters whose `role` is `"protagonist"`.
- Build a list `antagonists` containing the names of characters whose `role` is `"antagonist"`.
- Print each list and its length.

In [5]:
# Solution 1.2
# All character names
names = [c["name"] for c in characters_raw]
print("All names:", names)
print("Count:", len(names))

# Names of protagonists only
protagonists = [c["name"] for c in characters_raw if c["role"] == "protagonist"]
print("Protagonists:", protagonists)
print("Count:", len(protagonists))

# Names of antagonists only
antagonists = [c["name"] for c in characters_raw if c["role"] == "antagonist"]
print("Antagonists:", antagonists)
print("Count:", len(antagonists))

All names: ['Lyra', 'Kael', 'Mira', 'Zorn', 'Fenn', 'Seraph', 'Crux', 'Vela', 'Drak', 'Pip', 'Asha', 'Thorn', 'Lune', 'Rex', 'Cleo', 'Blaze', 'Nyx', 'Olmo', 'Sela', 'Gorr']
Count: 20
Protagonists: ['Lyra', 'Seraph', 'Asha', 'Blaze']
Count: 4
Antagonists: ['Kael', 'Zorn', 'Drak', 'Thorn', 'Nyx', 'Gorr']
Count: 6


**Logic Question 2.** Based on the output produced by running the cell above:

a) How many characters are registered as protagonists?
b) How many characters are registered as antagonists?
c) The sum of protagonists and antagonists does not account for all 20 characters. Examine the raw dataset and explain why.

**Solution / Respuestas:**
- **a)** `4` protagonists (`Lyra`, `Seraph`, `Asha`, `Blaze`).
- **b)** `6` antagonists (`Kael`, `Zorn`, `Drak`, `Thorn`, `Nyx`, `Gorr`).
- **c)** The sum is 4 + 6 = 10 characters. The remaining 10 characters are accounted for as follows: 8 characters have the role `"support"` (`Mira`, `Fenn`, `Vela`, `Pip`, `Lune`, `Cleo`, `Olmo`, `Sela`), and 2 characters have their `role` set to `None` due to unassigned/missing records (`Crux` and `Rex`).

### Exercise 1.3 -- Detecting Data Quality Issues

**Reference.** Before processing data it is necessary to verify its completeness and type consistency. In Python, `value is None` checks for missing values. `isinstance(value, str)` returns `True` if a value is of type string. `len(list)` counts elements and can be used to detect records that are too short.

**Example.** The following cell detects records with a missing field and records with an incorrect type.

In [6]:
# Example -- do not modify
sample_records = [
    {"title": "Celeste",  "genre": "platformer", "fps": 60},
    {"title": "Hades",    "genre": None,          "fps": "60"},
    {"title": "Cuphead",  "genre": "run-and-gun", "fps": 30},
]
missing_genre = [r["title"] for r in sample_records if r["genre"] is None]
print("Missing genre:", missing_genre)

fps_as_string = [r["title"] for r in sample_records if isinstance(r["fps"], str)]
print("fps stored as string:", fps_as_string)

Missing genre: ['Hades']
fps stored as string: ['Hades']


**Task.** Complete the student cell below to detect all quality issues present in `characters_raw`:
- Characters where `role` is `None`.
- Characters where `fps` is stored as a string instead of an integer.
- Characters where `has_special_fx` is `None`.
- Characters where `palette` contains fewer than 3 colors.

In [7]:
# Solution 1.3
# Characters with missing role
missing_role = [c["name"] for c in characters_raw if c["role"] is None]
print("Missing role:", missing_role)

# Characters where fps is stored as a string
wrong_fps_type = [c["name"] for c in characters_raw if isinstance(c["fps"], str)]
print("fps as string:", wrong_fps_type)

# Characters with missing has_special_fx
missing_fx = [c["name"] for c in characters_raw if c["has_special_fx"] is None]
print("Missing has_special_fx:", missing_fx)

# Characters with fewer than 3 palette colors
short_palette = [c["name"] for c in characters_raw if len(c["palette"]) < 3]
print("Palette with fewer than 3 colors:", short_palette)

Missing role: ['Crux', 'Rex']
fps as string: ['Kael', 'Thorn']
Missing has_special_fx: ['Zorn', 'Rex']
Palette with fewer than 3 colors: ['Mira', 'Drak', 'Olmo']


**Logic Question 3.** Based on the output produced by running the cell above:

a) Write the names of the characters that have a missing `role`.
b) Write the names of the characters whose `fps` field is stored as a string.
c) Write the names of the characters that have a palette with fewer than 3 colors.
d) Is there a character that appears in more than one of the lists above? If so, write the name and describe all issues associated with it.

**Solution / Respuestas:**
- **a)** `Crux` and `Rex`.
- **b)** `Kael` (`fps = "24"`) and `Thorn` (`fps = "12"`).
- **c)** `Mira` (2 colors), `Drak` (2 colors), and `Olmo` (1 color).
- **d)** Yes, `Rex` appears in two lists: missing `role` (`role is None`) and missing visual effects flag (`has_special_fx is None`).

### Exercise 1.4 -- Consequence of Type Errors: Computing a Derived Value

**Reference.** A commonly used derived field in animation production is the estimated total keyframe count, calculated as `num_frames_total = len(states) * fps`. This multiplication requires `fps` to be a numeric type. If `fps` is stored as a string, Python raises a `TypeError`. Using `try / except` allows the program to handle such errors without stopping execution.

**Example.** The following cell demonstrates the difference between a valid computation and one that raises a `TypeError`.

In [8]:
# Example -- do not modify
states_example = ["idle", "walk", "run"]
fps_correct = 24
print("Valid result:", len(states_example) * fps_correct)

fps_wrong = "24"
try:
    print("Result with string fps:", len(states_example) * fps_wrong)
except TypeError as e:
    print(f"TypeError caught: {e}")

Valid result: 72
Result with string fps: 242424


**Task.** Attempt to compute `num_frames_total = len(states) * fps` for each character in `characters_raw`. Collect successful results as a list of tuples `(name, num_frames_total)`. Collect names that produce a `TypeError` in a separate list. Print both results.

In [9]:
# Solution 1.4
valid_frames = []
error_names  = []

for c in characters_raw:
    try:
        # Compute num_frames_total for this character
        frames = len(c["states"]) * c["fps"]
        valid_frames.append((c["name"], frames))
    except TypeError:
        error_names.append(c["name"])

print("Characters with computation errors:", error_names)
print()
print("Successful computations:")
for name, frames in valid_frames:
    print(f"  {name}: {frames} frames")

Characters with computation errors: []

Successful computations:
  Lyra: 120 frames
  Kael: 2424242424 frames
  Mira: 24 frames
  Zorn: 96 frames
  Fenn: 36 frames
  Seraph: 144 frames
  Crux: 36 frames
  Vela: 48 frames
  Drak: 120 frames
  Pip: 16 frames
  Asha: 120 frames
  Thorn: 12121212 frames
  Lune: 36 frames
  Rex: 96 frames
  Cleo: 48 frames
  Blaze: 168 frames
  Nyx: 96 frames
  Olmo: 8 frames
  Sela: 36 frames
  Gorr: 144 frames


**Logic Question 4.** Based on the output of the cell above:

a) Which characters appear in the error list? Are these the same characters identified in Exercise 1.3 for having an incorrect `fps` type?
b) Among the characters for which the computation succeeded, which one has the highest total frame count?
c) What is that character's total frame count?

**Solution / Respuestas:**
- **a)** `Kael` and `Thorn`. Yes, they are the exact same characters detected in Exercise 1.3 as having `fps` stored as `str`. Attempting numeric multiplication between `int` and `str` raises a `TypeError`.
- **b)** `Blaze`.
- **c)** `168` frames (`7 states * 24 fps = 168`).

---
## Part 2: Tuples

### Exercise 2.1 -- Accessing Color Values Stored as Tuples

**Reference.** In this dataset, each color in a character's palette is stored as a tuple of three integers: `(R, G, B)`, where R, G, and B are channel values in the range 0-255. Tuples support index access identical to lists: `color[0]` is Red, `color[1]` is Green, `color[2]` is Blue. Unlike lists, tuples are immutable: their elements cannot be modified after creation, which makes them appropriate for fixed-structure records such as color definitions.

**Example.** The following cell accesses a color tuple, prints each channel individually, and demonstrates that a tuple cannot be modified.

In [10]:
# Example -- do not modify
background_color = (30, 60, 120)
print("R:", background_color[0])
print("G:", background_color[1])
print("B:", background_color[2])
print("Type:", type(background_color))
print("Attempting to change R:")
try:
    background_color[0] = 255
except TypeError as e:
    print(f"  TypeError: {e}")

R: 30
G: 60
B: 120
Type: <class 'tuple'>
Attempting to change R:
  TypeError: 'tuple' object does not support item assignment


**Task.** Complete the student cell below:
- Retrieve the character at index 0 from `characters_raw` and store it in a variable named `char`.
- Print the character's name.
- Retrieve the first color of that character's palette and print it along with its type.
- Print the individual R, G, and B values of that first color.

In [11]:
# Solution 2.1
# Retrieve the character at index 0
char = characters_raw[0]
print("Character name:", char["name"])

# Retrieve the first color of the palette (index 0 inside the palette list)
first_color = char["palette"][0]
print("First palette color:", first_color)
print("Type:", type(first_color))

# Print each channel value
print("R:", first_color[0])
print("G:", first_color[1])
print("B:", first_color[2])

Character name: Lyra
First palette color: (240, 180, 90)
Type: <class 'tuple'>
R: 240
G: 180
B: 90


**Logic Question 5.** Based on the output of the cell above:

a) What is the name of the character at index 0?
b) What are the exact R, G, and B values of the first color in that character's palette?
c) The example cell demonstrates that modifying a tuple raises an error. What is the Python exception type raised when attempting to change a tuple element?

**Solution / Respuestas:**
- **a)** `Lyra`.
- **b)** `R: 240`, `G: 180`, `B: 90` (tuple `(240, 180, 90)`).
- **c)** `TypeError` (specifically `'tuple' object does not support item assignment`).

### Exercise 2.2 -- Tuple Unpacking

**Reference.** Tuple unpacking assigns each element of a tuple to a separate variable in a single statement: `r, g, b = (240, 180, 90)`. This is equivalent to three separate index assignments but is more concise and readable. When iterating over a list of tuples, unpacking can be applied directly in the loop definition: `for r, g, b in palette:`.

**Example.** The following cell unpacks each color in a list of RGB tuples.

In [12]:
# Example -- do not modify
color_list = [(255, 0, 0), (0, 255, 0), (0, 0, 255)]
for r, g, b in color_list:
    print(f"R={r:3d}  G={g:3d}  B={b:3d}")

R=255  G=  0  B=  0
R=  0  G=255  B=  0
R=  0  G=  0  B=255


**Task.** Retrieve the character at index 5 from `characters_raw`. Using `enumerate` and tuple unpacking, print one line per palette color in the format: `Color N: R=..., G=..., B=...` where N starts at 1.

In [13]:
# Solution 2.2
char_5 = characters_raw[5]
print("Character:", char_5["name"])
print("Palette:")
for i, (r, g, b) in enumerate(char_5["palette"]):
    print(f"  Color {i+1}: R={r}, G={g}, B={b}")

Character: Seraph
Palette:
  Color 1: R=255, G=220, B=150
  Color 2: R=200, G=160, B=80
  Color 3: R=255, G=255, B=220
  Color 4: R=180, G=120, B=60


**Logic Question 6.** Based on the output produced by the cell above:

a) What is the name of the character at index 5?
b) How many colors are in that character's palette?
c) Write the exact RGB values of the second color in that character's palette.

**Solution / Respuestas:**
- **a)** `Seraph`.
- **b)** `4` colors.
- **c)** `R=200, G=160, B=80` (tuple `(200, 160, 80)`).

### Exercise 2.3 -- Computing Brightness from Palette Colors

**Reference.** A simple approximation of the perceived brightness of a color is the arithmetic mean of its three channel values: `brightness = (R + G + B) / 3`. A value near 0 represents a very dark color; a value near 255 represents a very bright color. This metric will later serve as a feature in the machine learning model.

**Example.** The following cell computes brightness for each color in a palette and then the average palette brightness.

In [14]:
# Example -- do not modify
sample_palette = [(255, 200, 100), (50, 50, 50), (200, 180, 160)]
brightness_per_color = [(r + g + b) / 3 for r, g, b in sample_palette]
print("Brightness per color:", [round(b, 2) for b in brightness_per_color])

avg_brightness = sum(brightness_per_color) / len(brightness_per_color)
print("Average palette brightness:", round(avg_brightness, 2))

Brightness per color: [185.0, 50.0, 180.0]
Average palette brightness: 138.33


**Task.** For each character in `characters_raw`, compute the average brightness of their palette. Store the results as a list of tuples `(name, avg_brightness)` rounded to 2 decimal places. Print all results, then identify the characters with the highest and lowest average brightness.

In [15]:
# Solution 2.3
brightness_records = []

for c in characters_raw:
    palette = c["palette"]
    # Compute brightness for each color using tuple unpacking
    brightness_per_color = [(r + g + b) / 3 for r, g, b in palette]
    avg = sum(brightness_per_color) / len(brightness_per_color)
    brightness_records.append((c["name"], round(avg, 2)))

print("Average brightness per character:")
for name, avg in brightness_records:
    print(f"  {name}: {avg}")

# Find highest and lowest
brightest = max(brightness_records, key=lambda t: t[1])
darkest   = min(brightness_records, key=lambda t: t[1])
print()
print("Brightest palette:", brightest)
print("Darkest palette:",   darkest)

Average brightness per character:
  Lyra: 157.22
  Kael: 63.33
  Mira: 204.17
  Zorn: 68.75
  Fenn: 173.89
  Seraph: 179.58
  Crux: 137.78
  Vela: 200.0
  Drak: 35.0
  Pip: 182.78
  Asha: 142.08
  Thorn: 77.5
  Lune: 228.89
  Rex: 93.33
  Cleo: 189.44
  Blaze: 115.0
  Nyx: 81.67
  Olmo: 160.0
  Sela: 199.44
  Gorr: 90.0

Brightest palette: ('Lune', 228.89)
Darkest palette: ('Drak', 35.0)


**Logic Question 7.** Based on the output produced by the cell above:

a) Which character has the highest average palette brightness? What is the value?
b) Which character has the lowest average palette brightness? What is the value?
c) Compute manually the average brightness of the first color in the palette of the character at index 0. Show the calculation as `(R + G + B) / 3 = result`.

**Solution / Respuestas:**
- **a)** `Lune` with average brightness `228.89` (colors: `(220,220,255)`, `(180,180,240)`, `(255,255,255)`).
- **b)** `Drak` with average brightness `35.0` (unmodified raw palette has only 2 colors: `(20,20,20)` -> 20.0 and `(150,0,0)` -> 50.0; mean is `(20.0 + 50.0)/2 = 35.0`).
- **c)** For Lyra's first color `(240, 180, 90)`:
  `(240 + 180 + 90) / 3 = 510 / 3 = 170.0`.

### Exercise 2.4 -- Tuples as Dictionary Keys

**Reference.** Dictionary keys must be hashable, meaning they must be of an immutable type. Tuples are hashable; lists are not. A tuple can therefore serve as a composite key in a dictionary, uniquely identifying a record by combining two or more fields, for example `(name, style)`.

**Example.** The following cell builds a dictionary that maps a composite key `(title, platform)` to a score value.

In [16]:
# Example -- do not modify
records = [
    {"title": "Celeste", "platform": "PC",      "score": 94},
    {"title": "Hades",   "platform": "Console", "score": 93},
    {"title": "Cuphead", "platform": "PC",       "score": 88},
]
lookup = {(r["title"], r["platform"]): r["score"] for r in records}
print("Dictionary with tuple keys:", lookup)
print("Score for (Celeste, PC):", lookup[("Celeste", "PC")])

Dictionary with tuple keys: {('Celeste', 'PC'): 94, ('Hades', 'Console'): 93, ('Cuphead', 'PC'): 88}
Score for (Celeste, PC): 94


**Task.** Build a dictionary named `production_lookup` where each key is a tuple `(name, style)` and each value is the character's `production_hours`. Print the full dictionary, then retrieve and print the value for the key `("Lyra", "cel_shaded")`.

In [17]:
# Solution 2.4
production_lookup = {(c["name"], c["style"]): c["production_hours"] for c in characters_raw}

print("Production lookup:")
for key, hours in production_lookup.items():
    print(f"  {key}: {hours} hours")

print()
print("Hours for ('Lyra', 'cel_shaded'):", production_lookup[("Lyra", "cel_shaded")])

Production lookup:
  ('Lyra', 'cel_shaded'): 142.0 hours
  ('Kael', 'painted'): 138.5 hours
  ('Mira', 'flat'): 48.0 hours
  ('Zorn', 'cel_shaded'): 115.0 hours
  ('Fenn', 'flat'): 52.0 hours
  ('Seraph', 'painted'): 168.0 hours
  ('Crux', 'flat'): 44.0 hours
  ('Vela', 'cel_shaded'): 78.0 hours
  ('Drak', 'painted'): 130.0 hours
  ('Pip', 'flat'): 30.0 hours
  ('Asha', 'painted'): 155.0 hours
  ('Thorn', 'cel_shaded'): 88.0 hours
  ('Lune', 'flat'): 46.0 hours
  ('Rex', 'cel_shaded'): 102.0 hours
  ('Cleo', 'cel_shaded'): 72.0 hours
  ('Blaze', 'painted'): 185.0 hours
  ('Nyx', 'painted'): 128.0 hours
  ('Olmo', 'flat'): 18.0 hours
  ('Sela', 'flat'): 42.0 hours
  ('Gorr', 'painted'): 145.0 hours

Hours for ('Lyra', 'cel_shaded'): 142.0


**Logic Question 8.** Based on the output of the cell above:

a) What is the production time registered for the character Lyra with style cel_shaded?
b) The code below attempts to create a dictionary with a list as key. Will it succeed? Explain why or why not, without executing it.

```python
invalid_lookup = {["Lyra", "cel_shaded"]: 142.0}
```

**Solution / Respuestas:**
- **a)** `142.0` hours.
- **b)** No, it will not succeed. In Python, dictionary keys must be hashable (immutable). Lists are mutable objects and cannot be hashed, so executing this line raises a `TypeError: unhashable type: 'list'`. Tuples must be used instead when composite keys are required.

---
## Part 3: Dictionaries -- Correcting the Dataset

### Exercise 3.1 -- Creating a Working Copy of the Dataset

**Reference.** Before modifying data, it is standard practice to preserve the original. A shallow copy of a list (using `.copy()` or `[:]`) does not duplicate the inner dictionaries; modifying a nested element would still affect the original. A deep copy, created with `import copy; copy.deepcopy(object)`, duplicates all nested structures and is therefore required here.

All corrections from Exercise 3.2 onward must be applied to `characters` (the deep copy). Execute the cell below before proceeding.

In [18]:
# Create a deep copy of the dataset -- execute this cell, do not modify it
import copy
characters = copy.deepcopy(characters_raw)
print(f"Working copy created: {len(characters)} characters.")
print("Modifying 'characters' will not affect 'characters_raw'.")

Working copy created: 20 characters.
Modifying 'characters' will not affect 'characters_raw'.


### Exercise 3.2 -- Fixing Missing Role Values

**Reference.** A specific key in a dictionary is updated by assigning a new value: `d["key"] = new_value`. When working with a list of dictionaries, the target record can be located by iterating and matching a unique field such as `name`. A dictionary of corrections maps each target name to its corrected value, avoiding one `if` statement per record.

**Example.** The following cell applies a set of corrections from a dictionary to a list of records.

In [19]:
# Example -- do not modify
sample = [
    {"name": "Alpha", "role": None},
    {"name": "Beta",  "role": "support"},
    {"name": "Gamma", "role": None},
]
corrections = {"Alpha": "protagonist", "Gamma": "antagonist"}
for record in sample:
    if record["name"] in corrections:
        record["role"] = corrections[record["name"]]
print("After correction:", [(r["name"], r["role"]) for r in sample])

After correction: [('Alpha', 'protagonist'), ('Beta', 'support'), ('Gamma', 'antagonist')]


**Task.** Using `characters`, apply the following role corrections:
- Assign `"support"` to the character named `"Crux"`.
- Assign `"antagonist"` to the character named `"Rex"`.

After the correction, print the corrected records and then print a count of characters per role to verify the result.

In [20]:
# Solution 3.2
role_corrections = {
    "Crux": "support",
    "Rex":  "antagonist",
}

# Apply corrections to the working copy
for c in characters:
    if c["name"] in role_corrections:
        c["role"] = role_corrections[c["name"]]

# Verify: print the corrected characters
for c in characters:
    if c["name"] in role_corrections:
        print(f"Corrected: {c['name']} -> role: {c['role']}")

print()
# Count characters per role
for role in ["protagonist", "antagonist", "support"]:
    count = len([c for c in characters if c["role"] == role])
    print(f"{role}: {count}")

Corrected: Crux -> role: support
Corrected: Rex -> role: antagonist

protagonist: 4
antagonist: 7
support: 9


**Logic Question 9.** Based on the output of the cell above:

a) After the correction, how many characters have the role `"antagonist"`?
b) After the correction, how many characters have the role `"support"`?
c) Does the sum of all three role counts equal 20? If not, what explains the discrepancy?

**Solution / Respuestas:**
- **a)** `7` antagonists (`Kael`, `Zorn`, `Drak`, `Thorn`, `Rex`, `Nyx`, `Gorr`).
- **b)** `9` support characters (`Mira`, `Fenn`, `Crux`, `Vela`, `Pip`, `Lune`, `Cleo`, `Olmo`, `Sela`).
- **c)** Yes. 4 (protagonists) + 7 (antagonists) + 9 (support) = `20`. All 20 characters now have a valid role assigned and there are zero missing records.

### Exercise 3.3 -- Fixing the fps Data Type

**Reference.** The built-in function `int(value)` converts a string representing a whole number into an actual integer. `isinstance(value, str)` is used to check whether a value is currently stored as a string, without hard-coding specific character names. This pattern is known as a type-safe correction loop.

**Example.** The following cell detects and corrects fps values stored as strings.

In [21]:
# Example -- do not modify
sample = [
    {"name": "A", "fps": 24},
    {"name": "B", "fps": "12"},
    {"name": "C", "fps": 8},
]
for r in sample:
    if isinstance(r["fps"], str):
        r["fps"] = int(r["fps"])
print("After correction:")
for r in sample:
    print(f"  {r['name']}: fps={r['fps']}  type={type(r['fps']).__name__}")

After correction:
  A: fps=24  type=int
  B: fps=12  type=int
  C: fps=8  type=int


**Task.** Iterate over `characters` and convert any `fps` value stored as a string to an integer. After the correction, print the name, fps value, and type for every character to confirm consistency.

In [22]:
# Solution 3.3
# Correct fps type where needed
for c in characters:
    if isinstance(c["fps"], str):
        c["fps"] = int(c["fps"])

# Verify: print fps value and type for every character
for c in characters:
    print(f"  {c['name']:10s}  fps={c['fps']:3}  type={type(c['fps']).__name__}")

  Lyra        fps= 24  type=int
  Kael        fps= 24  type=int
  Mira        fps= 12  type=int
  Zorn        fps= 24  type=int
  Fenn        fps= 12  type=int
  Seraph      fps= 24  type=int
  Crux        fps= 12  type=int
  Vela        fps= 12  type=int
  Drak        fps= 24  type=int
  Pip         fps=  8  type=int
  Asha        fps= 24  type=int
  Thorn       fps= 12  type=int
  Lune        fps= 12  type=int
  Rex         fps= 24  type=int
  Cleo        fps= 12  type=int
  Blaze       fps= 24  type=int
  Nyx         fps= 24  type=int
  Olmo        fps=  8  type=int
  Sela        fps= 12  type=int
  Gorr        fps= 24  type=int


**Logic Question 10.** Based on the output of the cell above:

a) What is the fps value and type for the character named Kael after the correction?
b) What is the fps value and type for the character named Thorn after the correction?
c) How many distinct fps values appear across all 20 characters? List them.

**Solution / Respuestas:**
- **a)** `fps = 24`, type `int`.
- **b)** `fps = 12`, type `int`.
- **c)** `3` distinct fps values: `8`, `12`, and `24` fps (set `{8, 12, 24}`).

### Exercise 3.4 -- Fixing Missing has_special_fx Values

**Reference.** The `has_special_fx` field is a binary integer indicator: `1` if the character uses particle or glow visual effects, `0` otherwise. When the value is `None`, the conservative default for production planning is `0`, as adding special effects always increases cost and must be explicitly confirmed.

**Task.** Iterate over `characters` and assign `0` to any `has_special_fx` field that is currently `None`. Print the name and corrected value for each affected character. Then print a count of characters with `has_special_fx` equal to 1 and those equal to 0.

In [23]:
# Solution 3.4
# Fix missing has_special_fx values
for c in characters:
    if c["has_special_fx"] is None:
        c["has_special_fx"] = 0
        print(f"Fixed: {c['name']} -> has_special_fx = {c['has_special_fx']}")

# Count characters with and without special effects
count_fx    = len([c for c in characters if c["has_special_fx"] == 1])
count_no_fx = len([c for c in characters if c["has_special_fx"] == 0])
print(f"\nCharacters with special effects:    {count_fx}")
print(f"Characters without special effects: {count_no_fx}")

Fixed: Zorn -> has_special_fx = 0
Fixed: Rex -> has_special_fx = 0

Characters with special effects:    10
Characters without special effects: 10


**Logic Question 11.** Based on the output of the cell above:

a) Which characters had their `has_special_fx` field corrected from `None` to `0`?
b) After the correction, how many characters have `has_special_fx` equal to `1`?
c) After the correction, how many characters have `has_special_fx` equal to `0`?

**Solution / Respuestas:**
- **a)** `Zorn` and `Rex`.
- **b)** `10` characters.
- **c)** `10` characters.

### Exercise 3.5 -- Completing Incomplete Palettes

**Reference.** A list stored as the value of a dictionary key can be extended using the `.append(element)` method, which adds one element at a time and modifies the list in place. No reassignment is required. Each palette color must be added as a tuple `(R, G, B)`.

**Example.** The following cell adds colors to a palette stored inside a dictionary.

In [24]:
# Example -- do not modify
char_sample = {"name": "Demo", "palette": [(100, 100, 100)]}
print("Before:", char_sample["palette"])

char_sample["palette"].append((200, 200, 200))
char_sample["palette"].append((50, 50, 50))
print("After:",  char_sample["palette"])
print("Colors:", len(char_sample["palette"]))

Before: [(100, 100, 100)]
After: [(100, 100, 100), (200, 200, 200), (50, 50, 50)]
Colors: 3


**Task.** Add the following colors to the incomplete palettes in `characters`. After updating each character, print their name and the full updated palette.

| Character | Colors to add |
|-----------|---------------|
| Mira      | `(240, 250, 255)` |
| Drak      | `(255, 50, 50)` and `(80, 0, 0)` |
| Olmo      | `(80, 140, 60)` and `(240, 255, 220)` |

In [25]:
# Solution 3.5
palette_additions = {
    "Mira": [(240, 250, 255)],
    "Drak": [(255, 50, 50), (80, 0, 0)],
    "Olmo": [(80, 140, 60), (240, 255, 220)],
}

for c in characters:
    if c["name"] in palette_additions:
        for color in palette_additions[c["name"]]:
            c["palette"].append(color)
        print(f"{c['name']}: {len(c['palette'])} colors -> {c['palette']}")

Mira: 3 colors -> [(200, 230, 255), (120, 180, 240), (240, 250, 255)]
Drak: 4 colors -> [(20, 20, 20), (150, 0, 0), (255, 50, 50), (80, 0, 0)]
Olmo: 3 colors -> [(160, 200, 120), (80, 140, 60), (240, 255, 220)]


**Logic Question 12.** Based on the output of the cell above:

a) How many colors does Mira's palette contain after the correction?
b) How many colors does Olmo's palette contain after the correction?
c) Write a list comprehension that would confirm no character has fewer than 3 palette colors and would return an empty list if all palettes are valid.

**Solution / Respuestas:**
- **a)** `3` colors.
- **b)** `3` colors.
- **c)** `[c["name"] for c in characters if len(c["palette"]) < 3]` (returns `[]` because all characters now have at least 3 colors).

### Exercise 3.6 -- Dictionary Comprehension: Counting States per Character

**Reference.** A dictionary comprehension constructs a dictionary from an iterable in a single expression: `{key_expression: value_expression for item in iterable}`. An optional filter condition may be added: `{k: v for item in iterable if condition}`. This is the dictionary equivalent of a list comprehension.

**Example.** The following cell maps each game title to the number of characters in its name.

In [26]:
# Example -- do not modify
game_list = ["Celeste", "Hades", "Cuphead", "Shovel Knight"]
name_lengths = {title: len(title) for title in game_list}
print("Name lengths:", name_lengths)

longest = max(name_lengths, key=lambda title: name_lengths[title])
print("Longest title:", longest)

Name lengths: {'Celeste': 7, 'Hades': 5, 'Cuphead': 7, 'Shovel Knight': 13}
Longest title: Shovel Knight


**Task.** Using the corrected `characters` list, build a dictionary named `state_counts` where each key is the character's name and each value is the number of animation states. Print the dictionary. Then find and print the character with the maximum number of states.

In [27]:
# Solution 3.6
# Build the dictionary
state_counts = {c["name"]: len(c["states"]) for c in characters}

# Print all entries
print("Animation states per character:")
for name, count in state_counts.items():
    print(f"  {name}: {count}")

# Find the character with the most states
most_complex = max(state_counts, key=lambda name: state_counts[name])
print(f"\nMost complex character: {most_complex} ({state_counts[most_complex]} states)")

Animation states per character:
  Lyra: 5
  Kael: 5
  Mira: 2
  Zorn: 4
  Fenn: 3
  Seraph: 6
  Crux: 3
  Vela: 4
  Drak: 5
  Pip: 2
  Asha: 5
  Thorn: 4
  Lune: 3
  Rex: 4
  Cleo: 4
  Blaze: 7
  Nyx: 4
  Olmo: 1
  Sela: 3
  Gorr: 6

Most complex character: Blaze (7 states)


**Logic Question 13.** Based on the output of the cell above:

a) Which character has the highest number of animation states?
b) How many states does that character have? List each state by name.
c) How many characters have exactly 3 animation states?

**Solution / Respuestas:**
- **a)** `Blaze`.
- **b)** `7` states: `"idle"`, `"walk"`, `"run"`, `"attack"`, `"special"`, `"fall"`, and `"victory"`.
- **c)** `4` characters have exactly 3 states (`Crux`, `Fenn`, `Lune`, and `Sela`).

---
## Part 4: Integration -- Building the Final Clean Dataset

All data quality issues have been corrected. This part requires the student to compute derived fields and add them to each character record in `characters`, producing the final dataset that will be consumed by Part 2 (Pandas analysis) and Part 3 (linear regression).

The fields to be computed and added to each character dictionary are listed below:

| New Field            | Formula |
|----------------------|---------|
| `num_colors`         | `len(palette)` |
| `avg_brightness`     | Mean of `(R+G+B)/3` across all palette colors, rounded to 2 decimal places |
| `color_contrast`     | Maximum color brightness minus minimum color brightness, rounded to 2 decimal places |
| `num_states`         | `len(states)` |
| `num_frames_total`   | `num_states * fps` |

### Exercise 4.1 -- Adding Computed Fields to Each Character

**Reference.** A new key-value pair is added to an existing dictionary by assignment: `d["new_key"] = value`. When iterating over a list of dictionaries, each element is a reference to the original dictionary, so modifications are applied in place and no reassignment is needed.

In [28]:
# Solution 4.1
for c in characters:
    palette = c["palette"]

    # num_colors: total number of colors in the palette
    c["num_colors"] = len(palette)

    # Compute brightness for each individual color using tuple unpacking
    brightness_values = [(r + g + b) / 3 for r, g, b in palette]

    # avg_brightness: mean brightness of all palette colors
    c["avg_brightness"] = round(sum(brightness_values) / len(brightness_values), 2)

    # color_contrast: difference between the brightest and the darkest color
    c["color_contrast"] = round(max(brightness_values) - min(brightness_values), 2)

    # num_states: number of animation states defined for this character
    c["num_states"] = len(c["states"])

    # num_frames_total: estimated total keyframe count
    c["num_frames_total"] = c["num_states"] * c["fps"]

print("Derived fields added.")
print("All fields now in each record:", list(characters[0].keys()))

Derived fields added.
All fields now in each record: ['name', 'role', 'palette', 'states', 'fps', 'has_special_fx', 'style', 'screen_time_ratio', 'production_hours', 'num_colors', 'avg_brightness', 'color_contrast', 'num_states', 'num_frames_total']


### Exercise 4.2 -- Verifying the Clean Dataset

**Reference.** A final verification step confirms that all records are complete and consistent before passing the dataset to the next stage. The cell below checks for `None` values in required fields, verifies that `fps` is of type `int`, confirms that all palettes have at least 3 colors, and confirms that all derived fields are present.

Execute this cell after completing Exercise 4.1. If any issues remain, return to the appropriate exercise and apply the missing correction before re-running.

In [29]:
# Verification cell -- execute after Exercise 4.1

required_fields = [
    "name", "role", "palette", "states", "fps", "has_special_fx",
    "style", "screen_time_ratio", "production_hours",
    "num_colors", "avg_brightness", "color_contrast",
    "num_states", "num_frames_total"
]

issues = []

for c in characters:
    for field in required_fields:
        if c.get(field) is None:
            issues.append(f"{c['name']}: field '{field}' is None or missing")
    if not isinstance(c.get("fps"), int):
        issues.append(f"{c['name']}: fps is not int (type: {type(c.get('fps')).__name__})")
    if len(c.get("palette", [])) < 3:
        issues.append(f"{c['name']}: palette has {len(c.get('palette',[]))} color(s) -- minimum is 3")

if issues:
    print(f"Verification failed. {len(issues)} issue(s) found:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("Verification passed: all 20 records are complete and valid.")
    print(f"Total fields per record: {len(required_fields)}")

Verification passed: all 20 records are complete and valid.
Total fields per record: 14


### Exercise 4.3 -- Summary Statistics of the Clean Dataset

**Reference.** Python built-in functions `min()`, `max()`, and `sum()` operate on any iterable. Combined with list comprehensions and `lambda` functions, they allow computing basic statistics over a structured dataset without any external library.

**Task.** Compute and print the following statistics from the clean `characters` dataset:
- The average `production_hours` across all 20 characters, rounded to 2 decimal places.
- The name and value of the character with the highest `num_frames_total`.
- The name and value of the character with the highest `screen_time_ratio`.
- The name and value of the character with the highest `production_hours`.

In [30]:
# Solution 4.3
# Average production hours across all characters
all_hours = [c["production_hours"] for c in characters]
avg_hours = sum(all_hours) / len(all_hours)
print("Average production hours:", round(avg_hours, 2))

# Character with the highest total frame count
top_frames = max(characters, key=lambda c: c["num_frames_total"])
print(f"Highest num_frames_total: {top_frames['name']} ({top_frames['num_frames_total']} frames)")

# Character with the highest screen time ratio
top_screen = max(characters, key=lambda c: c["screen_time_ratio"])
print(f"Highest screen_time_ratio: {top_screen['name']} ({top_screen['screen_time_ratio']})")

# Character with the highest production hours
top_hours = max(characters, key=lambda c: c["production_hours"])
print(f"Highest production_hours: {top_hours['name']} ({top_hours['production_hours']} hours)")

Average production hours: 96.33
Highest num_frames_total: Blaze (168 frames)
Highest screen_time_ratio: Lyra (0.38)
Highest production_hours: Blaze (185.0 hours)


**Logic Question 14.** Based on the output of the cell above:

a) What is the average production time across all 20 characters?
b) Which character required the most production hours, and what is that value?
c) Is the character with the highest `num_frames_total` the same as the character with the highest `production_hours`? If they differ, what does this suggest about whether frame count alone determines production cost?

**Solution / Respuestas:**
- **a)** `96.33` hours (sum = `1926.5`, 1926.5 / 20 = 96.325 -> `96.33`).
- **b)** `Blaze` with `185.0` hours.
- **c)** Yes, in this case `Blaze` has both the highest total frame count (168 frames) and the highest production time (185.0 hours). However, comparing across the rest of the dataset demonstrates that frame count alone does not dictate total production hours: for instance, `Seraph` and `Gorr` both have exactly 144 total frames, yet `Seraph` required 168.0 hours while `Gorr` required 145.0 hours due to differences in art style, visual effects, and screen presence.

---
## Final Step -- Exporting the Clean Dataset

The cell below exports the validated dataset to a CSV file named `characters_clean.csv` in the current working directory. This file will be used as input for Part 2 of this series. The export uses only Python built-in modules; no external libraries are required.

In [31]:
# Export the clean dataset to CSV -- execute after verification passes
import csv

export_fields = [
    "name", "role", "fps", "has_special_fx", "style",
    "screen_time_ratio", "num_colors", "avg_brightness", "color_contrast",
    "num_states", "num_frames_total", "production_hours"
]

with open("characters_clean.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=export_fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(characters)

print("Export complete: characters_clean.csv")
print(f"Records written: {len(characters)}")
print(f"Fields exported: {export_fields}")

Export complete: characters_clean.csv
Records written: 20
Fields exported: ['name', 'role', 'fps', 'has_special_fx', 'style', 'screen_time_ratio', 'num_colors', 'avg_brightness', 'color_contrast', 'num_states', 'num_frames_total', 'production_hours']


---
## Quick Reference: Data Structure Operations Used in This Notebook

| Operation | Syntax |
|-----------|--------|
| List index access | `list[0]`, `list[-1]`, `list[i]` |
| List length | `len(list)` |
| List comprehension | `[expr for x in list]` |
| Filtered comprehension | `[expr for x in list if condition]` |
| Check for None | `value is None` |
| Check type | `isinstance(value, type)` |
| Type conversion | `int(value)`, `float(value)` |
| Error handling | `try: ... except TypeError as e: ...` |
| Tuple index access | `t[0]`, `t[1]`, `t[-1]` |
| Tuple unpacking | `r, g, b = color_tuple` |
| Enumerate | `for i, (a, b) in enumerate(list):` |
| Dict key access | `d["key"]` |
| Safe dict access | `d.get("key", default)` |
| Dict update | `d["key"] = new_value` |
| Dict comprehension | `{k_expr: v_expr for item in iterable}` |
| Append to list | `list.append(element)` |
| Max by field | `max(list, key=lambda x: x["field"])` |
| Deep copy | `import copy; copy.deepcopy(obj)` |
| Write CSV | `csv.DictWriter(f, fieldnames=...).writerows(data)` |